In [ ]:
import sys
print("🔧 Instalando dependencias: pydantic")
!{sys.executable} -m pip install -q pydantic
print("✅ Instalación completada. Ejecuta las celdas siguientes.")

# AEM4L1 E10 — Avanzado | Golden Cases Visuales

**Objetivo:** Evaluar outputs con `valid_schema`, `accuracy` y `completeness`, y entender el flujo de Golden Cases.

**Tipo:** Resuelto

**Uso:** listo para Google Colab. Ejecutá las celdas en orden.

## 📦 Dependencias

- `pydantic`

**Cómo instalar:**
```bash
pip install pydantic
```

La celda de instalación ya está incluida arriba. Ejecutala primero.

## 🧠 ¿Qué es un Golden Case?

Un Golden Case es un ejemplo de salida esperada (`expected`) que usamos como referencia para evaluar la salida real (`actual`) de un modelo o pipeline.

- `expected`: JSON correcto que debería producir el sistema.
- `actual`: JSON generado por el modelo/pipeline.

Comparar ambos nos permite detectar errores de contenido, formatos y estructura.

## 📊 Métricas de evaluación

1. `valid_schema` — comprueba que el JSON `actual` cumpla con el esquema definido en `DocumentOutput`.
2. `accuracy` — porcentaje de campos cuyos valores coinciden exactamente con el `expected`.
3. `completeness` — porcentaje de campos esperados que aparecen en el `actual`.

### Diagrama de flujo

```text
[ Modelo / Pipeline ]
           ↓
     Actual JSON
           ↓
  Comparar con Expected
           ↓
 Métricas: schema, accuracy, completeness
```

### Tabla de métricas

Propósito | ¿Qué mide? | Ejemplo
--- | --- | ---
`valid_schema` | Valida tipos y campos requeridos | `True` / `False`
`accuracy` | Campos con valor correcto | `0.75`
`completeness` | Campos presentes | `0.80`

In [ ]:
from pydantic import BaseModel, ValidationError
from datetime import date

class DocumentOutput(BaseModel):
    document_type: str
    issuer: str
    date: date
    amount: float
    is_valid: bool

golden_cases = [
    {
        "name": "Perfecto",
        "expected": {
            "document_type": "ticket",
            "issuer": "Supermercado Norte",
            "date": "2026-06-12",
            "amount": 8500,
            "is_valid": True
        },
        "actual": {
            "document_type": "ticket",
            "issuer": "Supermercado Norte",
            "date": "2026-06-12",
            "amount": 8500,
            "is_valid": True
        }
    },
    {
        "name": "Valor incorrecto",
        "expected": {
            "document_type": "factura",
            "issuer": "Servicios Delta",
            "date": "2026-06-10",
            "amount": 15000,
            "is_valid": True
        },
        "actual": {
            "document_type": "factura",
            "issuer": "Servicios Delta",
            "date": "2026-06-10",
            "amount": 14000,
            "is_valid": True
        }
    },
    {
        "name": "Formato inválido",
        "expected": {
            "document_type": "ticket",
            "issuer": "Cine Total",
            "date": "2026-06-09",
            "amount": 12000,
            "is_valid": True
        },
        "actual": {
            "document_type": "ticket",
            "issuer": "Cine Total",
            "date": "09-06-2026",
            "amount": 12000,
            "is_valid": True
        }
    }
]


def evaluate(expected, actual):
    try:
        DocumentOutput(**actual)
        valid_schema = True
    except ValidationError:
        valid_schema = False

    total = len(expected)
    present = sum(1 for k in expected if k in actual)
    correct = sum(1 for k in expected if actual.get(k) == expected[k])

    return {
        "valid_schema": valid_schema,
        "accuracy": correct / total,
        "completeness": present / total
    }


def print_results(cases):
    header = f"{'Caso':<18} | {'valid_schema':<12} | {'accuracy':<8} | {'completeness':<11}"
    separator = '-' * len(header)
    print(header)
    print(separator)
    for case in cases:
        metrics = evaluate(case["expected"], case["actual"])
        print(
            f"{case['name']:<18} | {str(metrics['valid_schema']):<12} | "
            f"{metrics['accuracy']:<8.2f} | {metrics['completeness']:<11.2f}"
        )
        if metrics['valid_schema'] is False:
            print("  → El JSON actual no cumple el esquema Pydantic. Revisa tipos/formatos.")
    print(separator)

print_results(golden_cases)

## ✅ Conclusión

- Un Golden Case perfecto debe tener `valid_schema = True`, `accuracy = 1.00` y `completeness = 1.00`.
- Cuando un valor difiere, `accuracy` baja pero el esquema puede seguir siendo válido.
- Cuando el formato o el tipo es incorrecto, `valid_schema` se vuelve `False`.

Usa este enfoque para revisar outputs reales y detectar rápidamente errores en estructura y contenido.